In [11]:
# === CELL 1: IMPORTS AND CONFIGURATION ===
import subprocess
subprocess.run(["pip", "install", "undetected-chromedriver"], capture_output=True)

import time
import random
import os
import re
import json
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

OUTPUT_CSV = "data/entrackr_complaints.csv"
OUTPUT_JSON = "data/entrackr.json"
TXT_DIR = "data/txt"
MAX_PAGES = 10
BASE_URL = "https://entrackr.com"
os.makedirs(TXT_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

def init_driver():
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = uc.Chrome(options=options, headless=True, version_main=None)
    driver.execute_cdp_cmd("Network.setUserAgentOverride", {
        "userAgent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    })
    return driver

# ── ID Continuity ──────────────────────────────────────────────────────────
existing_max_id = 0
for csv_path in [
    "data/consumer_complaints.csv",
    "data/indiankanoon_complaints.csv",
    "data/medianama_complaints.csv",
    "data/reddit_complaints.csv",
    "data/inc42_complaints.csv",
]:
    if os.path.exists(csv_path):
        df_check = pd.read_csv(csv_path)
        if "Unique ID" in df_check.columns:
            ids = df_check["Unique ID"].dropna().str.extract(r'(\d+)')[0].astype(float)
            if not ids.empty:
                existing_max_id = max(existing_max_id, int(ids.max()))

# Also scan txt folder directly as safety net
for fname in os.listdir("data/txt"):
    match = re.search(r'NA-(\d+)\.txt', fname)
    if match:
        existing_max_id = max(existing_max_id, int(match.group(1)))

next_id = existing_max_id + 1
print(f"Starting Unique ID from: NA-{next_id:04d}")

Starting Unique ID from: NA-28130


In [12]:
# === CELL 2: KEYWORD TAXONOMY ===
KEYWORD_TAXONOMY = {
    "General Cybercrime / Cyber Fraud Terms": {
        "General Cybercrime": ["cyber crime", "cybercrime", "cyber fraud", "online fraud", "internet fraud", "digital fraud"],
        "Cyber Scam": ["cyber scam", "online scam", "internet scam", "online cheating"],
        "Financial Cyber Fraud": ["financial fraud online", "net banking fraud", "e-banking fraud"]
    },
    "UPI and Digital Payment Fraud": {
        "UPI Fraud": ["UPI fraud", "UPI scam", "Google Pay fraud", "PhonePe fraud", "Paytm fraud", "BHIM fraud"],
        "QR Code Fraud": ["QR code scam", "QR code fraud", "scan and pay fraud"],
        "Payment Link Fraud": ["payment link fraud", "collect request scam", "fake payment link"],
        "Mobile Wallet Fraud": ["mobile wallet fraud", "e-wallet scam", "digital wallet fraud"]
    },
    "OTP and Authentication Fraud": {
        "OTP Fraud": ["OTP fraud", "OTP scam", "OTP theft"],
        "SIM Swap": ["SIM swap fraud", "SIM cloning", "duplicate SIM fraud"],
        "KYC Fraud": ["KYC fraud", "KYC scam", "Aadhaar KYC scam"]
    },
    "Digital Arrest Scam": {
        "Digital Arrest": ["digital arrest", "digital arrest scam", "fake arrest", "video call arrest"],
        "Impersonation Scam": ["police impersonation scam", "CBI fraud call", "customs fraud call", "TRAI scam call", "ED scam call"],
        "Video Call Coercion": ["video call scam", "video call blackmail", "video call extortion", "fake interrogation"]
    },
    "Phishing, Vishing, and Smishing": {
        "Phishing": ["phishing", "phishing attack", "phishing email", "fake website", "spoof website"],
        "Vishing": ["vishing", "voice phishing", "fraud call", "fake bank call"],
        "Smishing": ["smishing", "SMS fraud", "SMS scam", "phishing SMS"]
    },
    "Online Lending and Loan App Fraud": {
        "Loan App Fraud": ["loan app fraud", "instant loan scam", "loan app harassment", "illegal loan app"],
        "Loan App Extortion": ["loan app blackmail", "loan app threat", "morphed photos loan", "recovery agent threat"]
    },
    "Investment and Trading Fraud": {
        "Investment Scam": ["investment scam", "Ponzi scheme", "online investment fraud", "crypto scam", "bitcoin fraud", "forex trading scam", "pig butchering"],
        "Stock Market Fraud": ["stock market scam", "share trading fraud", "demat fraud", "pump and dump"],
        "Task Scam": ["task fraud", "part time job scam", "work from home scam", "Telegram task scam"]
    },
    "Identity Theft and Data Breach": {
        "Identity Theft": ["identity theft", "identity fraud", "Aadhaar misuse", "PAN fraud"],
        "Data Breach": ["data breach", "data leak", "data theft", "customer data breach"]
    },
    "Social Engineering and Romance/Sextortion": {
        "Social Engineering": ["social engineering fraud", "manipulation scam", "trust scam"],
        "Romance Scam": ["romance scam", "dating fraud", "matrimonial fraud", "honey trap", "catfishing fraud"],
        "Sextortion": ["sextortion", "webcam blackmail", "nude video blackmail"]
    },
    "E-Commerce and Delivery Fraud": {
        "E-Commerce Fraud": ["e-commerce fraud", "online shopping fraud", "fake product scam", "Flipkart fraud", "Amazon fraud", "refund scam"],
        "Delivery Fraud": ["fake delivery", "courier fraud", "customs duty scam", "parcel scam", "delivery OTP scam"]
    },
    "Ransomware and Malware": {
        "Ransomware": ["ransomware attack", "cyber ransom", "data encryption attack"],
        "Banking Malware": ["banking trojan", "banking malware", "keylogger fraud", "AnyDesk fraud", "TeamViewer scam", "screen sharing scam"]
    },
    "Emerging and Miscellaneous Fraud Types": {
        "Deepfake Fraud": ["deepfake scam", "deepfake fraud", "AI voice scam", "voice cloning scam"],
        "Utility Scam": ["electricity bill scam", "utility fraud", "disconnection scam"],
        "Aadhaar Fraud": ["Aadhaar fraud", "Aadhaar scam", "biometric fraud", "AEPS fraud"],
        "Cyber Stalking": ["cyber stalking", "cyber bullying", "online harassment", "digital harassment"]
    }
}

In [13]:
# === CELL 3: HELPER FUNCTIONS ===
def classify_narrative_type(text):
    text_lower = text.lower()
    if any(x in text_lower for x in ["victim", "lost", "cheated", "defrauded",
                                       "money stolen", "account hacked", "fell for"]):
        return "VICTIM"
    elif any(x in text_lower for x in ["beware", "warning", "alert", "avoid",
                                         "do not", "scam alert", "red flag"]):
        return "NEAR-MISS"
    return "THIRD-PARTY"

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def safe_save(df, csv_path, json_path, json_data):
    try:
        temp_csv = csv_path + ".tmp"
        df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        os.replace(temp_csv, csv_path)
        temp_json = json_path + ".tmp"
        with open(temp_json, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=4, ensure_ascii=False)
        os.replace(temp_json, json_path)
        print(f"  ✅ Checkpoint saved: {len(df)} records")
    except Exception as e:
        print(f"  ⚠️ Checkpoint save failed (data still in memory): {e}")

In [14]:
# === CELL 4: SELENIUM SEARCH FUNCTION ===
def search_entrackr_selenium(driver, keyword, max_pages=10):
    results = []
    seen_urls = set()

    for page in range(1, max_pages + 1):
        if page == 1:
            search_url = f"https://entrackr.com/search?title={keyword.replace(' ', '+')}"
        else:
            search_url = f"https://entrackr.com/search?title={keyword.replace(' ', '+')}&page={page}"
            
        print(f"  Loading page {page}: {search_url}")
        
        try:
            driver.get(search_url)
            time.sleep(random.uniform(5.0, 8.0))
            
            # Verify we passed Cloudflare
            if "just a moment" in driver.page_source.lower() or "enable javascript" in driver.page_source.lower():
                print(f"  Cloudflare challenge on page {page} — waiting longer...")
                time.sleep(random.uniform(8.0, 12.0))
                if "just a moment" in driver.page_source.lower():
                    print(f"  Still blocked — skipping this page")
                    break
            
            soup = BeautifulSoup(driver.page_source, 'html.parser')

            # Print page title for debugging
            title_tag = soup.find('title')
            print(f"  Page {page} title: {title_tag.get_text() if title_tag else 'Unknown'}")

            article_divs = soup.find_all('div', class_='feat-a-1')
            print(f"  Page {page}: {len(article_divs)} article divs found")

            if not article_divs:
                print("  No articles found. Stopping pagination.")
                break

            page_results = 0
            for div in article_divs:
                all_links = div.find_all('a', href=True)
                for link_tag in all_links:
                    href = link_tag.get('href', '')
                    if not href or href == '#':
                        continue

                    full_url = "https://entrackr.com" + href if href.startswith('/') else href
                    if 'entrackr.com' not in full_url:
                        continue
                    if full_url in seen_urls:
                        continue

                    # Get title — from aria-label of THIS specific link or nearest heading
                    title = link_tag.get('aria-label', '').strip()
                    if not title:
                        heading = div.find(['h1', 'h2', 'h3', 'h4'])
                        title = heading.get_text(strip=True) if heading else link_tag.get_text(strip=True)

                    if len(title) < 10:
                        continue

                    seen_urls.add(full_url)

                    # Fix date extraction
                    date = "Unknown Date"
                    time_tag = div.find('time')
                    if time_tag and time_tag.get('datetime'):
                        date = time_tag['datetime'][:10]
                    else:
                        for el in div.find_all(['span', 'div', 'p']):
                            text = el.get_text(strip=True)
                            # Match YYYY-MM-DD or DD Mon YYYY
                            m = re.search(r'(\d{4}-\d{2}-\d{2})', text)
                            if m:
                                date = m.group(1)
                                break
                            m2 = re.search(r'(\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{4})', text)
                            if m2:
                                date = m2.group(1)
                                break

                    results.append({"title": title, "url": full_url, "date": date})
                    page_results += 1
                    break  # one result per feat-a-1 div

            print(f"  Page {page}: added {page_results} results (total: {len(results)})")

            if page_results == 0:
                print(f"  No new results on page {page} — stopping pagination")
                break

        except Exception as e:
            print(f"  Error on page {page}: {e}")
            break

    return results

In [15]:
# === CELL 5: ARTICLE FULL TEXT FETCH FUNCTION ===
def fetch_entrackr_article(driver, url):
    try:
        driver.get(url)
        time.sleep(random.uniform(4.0, 6.0))
        if "just a moment" in driver.page_source.lower():
            time.sleep(8)

        # Wait for article content to load
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "article"))
            )
        except TimeoutException:
            pass  # continue anyway and try to parse what loaded

        time.sleep(random.uniform(2.0, 4.0))
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Remove noise
        for tag in soup.find_all(['script', 'style', 'aside',
                                   'figure', 'nav', 'iframe']):
            tag.decompose()
        for cls in ['related-posts', 'social-share', 'newsletter',
                     'sidebar', 'comments', 'advertisement']:
            for el in soup.find_all(class_=re.compile(cls, re.I)):
                el.decompose()

        # Get article body — try multiple selectors in priority order
        content = (
            soup.find('div', class_=re.compile(r'entry.content|post.content|article.content|story.content', re.I)) or
            soup.find('div', class_='content') or
            soup.find('article') or
            soup.find('main')
        )

        text = clean_text(content.get_text(separator='\n', strip=True)) if content else ""

        # Get author
        author_tag = (
            soup.find('a', rel='author') or
            soup.find(class_=re.compile(r'author.name|author', re.I)) or
            soup.find('span', class_='author')
        )
        author = author_tag.get_text(strip=True) if author_tag else "Unknown"

        # Get date
        time_tag = soup.find('time')
        if time_tag and time_tag.get('datetime'):
            date = time_tag['datetime'][:10]
        else:
            # Search all text nodes for a clean date pattern
            full_text_body = soup.get_text()
            m = re.search(r'(\d{4}-\d{2}-\d{2})', full_text_body)
            if m:
                date = m.group(1)
            else:
                m2 = re.search(r'(\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{4})', full_text_body)
                date = m2.group(1) if m2 else "Unknown Date"

        return text, author, date

    except TimeoutException:
        print(f"  Timeout: {url}")
        return "", "Unknown", "Unknown Date"
    except Exception as e:
        print(f"  Error fetching {url}: {e}")
        return "", "Unknown", "Unknown Date"

In [17]:
# === CELL 6: MAIN SCRAPING LOOP ===
# Load existing data to avoid re-scraping
existing_urls = set()
existing_data = []
if os.path.exists(OUTPUT_JSON):
    try:
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            existing_data = json.load(f)
        for item in existing_data:
            if item.get("URL"):
                existing_urls.add(item["URL"])
    except json.JSONDecodeError:
        existing_data = []
print(f"Loaded {len(existing_urls)} existing URLs to skip")

today_date = datetime.now().strftime("%Y-%m-%d")
all_new_records = []
all_new_raw = []
new_count = 0

driver = init_driver()

try:
    for parent_category, subcategories in KEYWORD_TAXONOMY.items():
        for subcat, keywords in subcategories.items():
            for keyword in keywords:
                print(f"\n[*] '{keyword}' | {parent_category} > {subcat}")

                search_results = search_entrackr_selenium(driver, keyword, max_pages=MAX_PAGES)

                for res in search_results:
                    url = res['url']
                    if url in existing_urls:
                        continue
                    existing_urls.add(url)

                    print(f"  Fetching article: {url[:80]}")
                    full_text, author, date = fetch_entrackr_article(driver, url)

                    # Use search result date if article date not found
                    if date == "Unknown Date" and res['date'] != "Unknown Date":
                        date = res['date']

                    title = res['title']
                    unique_id = f"NA-{next_id:04d}"
                    txt_filename = f"{unique_id}.txt"

                    # Save txt file immediately
                    txt_content = f"SOURCE: Entrackr\n"
                    txt_content += f"TITLE: {title}\n"
                    txt_content += f"AUTHOR: {author}\n"
                    txt_content += f"DATE: {date}\n"
                    txt_content += f"URL: {url}\n\n"
                    txt_content += "--- ARTICLE TEXT ---\n\n"
                    txt_content += full_text if full_text else "[Article text unavailable]"

                    with open(os.path.join(TXT_DIR, txt_filename), "w", encoding="utf-8") as f:
                        f.write(txt_content)

                    # Build record — exact same fields as all other scrapers
                    record = {
                        "Unique ID": unique_id,
                        "Date of Collection": today_date,
                        "Collector Name": "Soubhik Sarkar",
                        "Source Platform": "Entrackr",
                        "Source Publication": "entrackr.com",
                        "Original Date": date,
                        "Title/Headline": title,
                        "URL": url,
                        "Search Query Used": keyword,
                        "Fraud Category": parent_category,
                        "Fraud Subcategory": subcat,
                        "Narrative Type": classify_narrative_type(full_text),
                        "TXT File Name": txt_filename,
                        "Notes": f"Author: {author} | Category: {res.get('category_label', 'News')}"
                    }

                    raw_item = {
                        "URL": url,
                        "Title/Headline": title,
                        "Original Date": date,
                        "Author": author,
                        "StructuredData": record
                    }

                    all_new_records.append(record)
                    all_new_raw.append(raw_item)
                    next_id += 1
                    new_count += 1

                    # Checkpoint every 25 records
                    if new_count % 25 == 0:
                        df_temp = pd.DataFrame(all_new_records)
                        combined_raw = existing_data + all_new_raw
                        safe_save(df_temp, OUTPUT_CSV, OUTPUT_JSON, combined_raw)

finally:
    driver.quit()
    print(f"\nDriver closed. Total new records this session: {new_count}")

Loaded 0 existing URLs to skip


KeyboardInterrupt: 

In [ ]:
# === CELL 7: FINAL SAVE ===
if all_new_records:
    df_new = pd.DataFrame(all_new_records)

    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        df_combined.drop_duplicates(subset=["URL"], keep="last", inplace=True)
    else:
        df_combined = df_new

    temp_csv = OUTPUT_CSV + ".tmp"
    df_combined.to_csv(temp_csv, index=False, encoding="utf-8-sig")
    os.replace(temp_csv, OUTPUT_CSV)
    df_combined.to_excel(OUTPUT_CSV.replace(".csv", ".xlsx"), index=False)

    combined_raw = existing_data + all_new_raw
    json_dedup = {v["URL"]: v for v in combined_raw}
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(list(json_dedup.values()), f, indent=4, ensure_ascii=False)

    ids = df_combined["Unique ID"].dropna().str.extract(r'(\d+)')[0].astype(float)
    print(f"✅ Saved {len(df_combined)} total records")
    print(f"IDs: NA-{int(ids.min()):04d} to NA-{int(ids.max()):04d}")
    print(f"CSV: {OUTPUT_CSV}")
    print(f"JSON: {OUTPUT_JSON}")
    print(f"TXT files: {TXT_DIR}/")
else:
    print("No new records to save.")

In [16]:
# === CELL 8: DIAGNOSTIC CELL (run this first before main loop) ===
driver = init_driver()
try:
    # Test search page
    driver.get("https://entrackr.com/search?title=UPI+fraud")
    time.sleep(5)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    divs = soup.find_all('div', class_='feat-a-1')
    print(f"feat-a-1 divs found: {len(divs)}")

    if divs:
        for div in divs[:3]:
            link = div.find('a', class_='clickable') or div.find('a', href=True)
            if link:
                title = link.get('aria-label', link.get_text(strip=True))[:100]
                href = link.get('href', '')
                full_url = "https://entrackr.com" + href if href.startswith('/') else href
                print(f"\nTitle: {title}")
                print(f"URL: {full_url}")

        # Test article fetch on first result
        first_link = divs[0].find('a', class_='clickable') or divs[0].find('a', href=True)
        if first_link:
            article_url = "https://entrackr.com" + first_link.get('href', '')
            print(f"\nFetching article: {article_url}")
            text, author, date = fetch_entrackr_article(driver, article_url)
            print(f"Author: {author}")
            print(f"Date: {date}")
            print(f"Text length: {len(text)}")
            print(f"Text preview:\n{text[:400]}")

    # Test pagination
    print("\n=== Testing pagination ===")
    driver.get("https://entrackr.com/search?title=UPI+fraud&page=2")
    time.sleep(4)
    page2_url = driver.current_url
    page2_soup = BeautifulSoup(driver.page_source, 'html.parser')
    page2_divs = page2_soup.find_all('div', class_='feat-a-1')
    print(f"Page 2 URL: {page2_url}")
    print(f"Page 2 results: {len(page2_divs)}")

finally:
    driver.quit()
    print("\nDiagnostic complete.")

SessionNotCreatedException: Message: session not created: cannot connect to chrome at 127.0.0.1:55064
from session not created: This version of ChromeDriver only supports Chrome version 147
Current browser version is 146.0.7680.165; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
	undetected_chromedriver!GetHandleVerifier [0x45c6f3+10b73]
	undetected_chromedriver!GetHandleVerifier [0x45c824+10ca4]
	undetected_chromedriver!(No symbol) [0x232100]
	undetected_chromedriver!(No symbol) [0x26d752]
	undetected_chromedriver!(No symbol) [0x26c77c]
	undetected_chromedriver!(No symbol) [0x2628a5]
	undetected_chromedriver!(No symbol) [0x2626c6]
	undetected_chromedriver!(No symbol) [0x2a9dff]
	undetected_chromedriver!(No symbol) [0x2a9617]
	undetected_chromedriver!(No symbol) [0x29d9b6]
	undetected_chromedriver!(No symbol) [0x270339]
	undetected_chromedriver!(No symbol) [0x2710f4]
	undetect

In [10]:
driver = init_driver()
try:
    driver.get("https://entrackr.com/search?title=UPI+fraud")
    time.sleep(4)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    divs = soup.find_all('div', class_='feat-a-1')
    print(f"Results found: {len(divs)}")
    for div in divs[:3]:
        link = div.find('a', href=True)
        if link:
            print(f"Title: {link.get('aria-label', link.get_text(strip=True))[:100]}")
            print(f"URL: {link.get('href')}")
finally:
    driver.quit()

Results found: 0


In [8]:
driver = init_driver()
try:
    driver.get("https://entrackr.com")
    time.sleep(3)
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    # Print ALL input elements
    print("=== All inputs ===")
    for inp in soup.find_all('input'):
        print(f"  type={inp.get('type')} | id={inp.get('id')} | class={inp.get('class')} | name={inp.get('name')} | placeholder={inp.get('placeholder')}")
    
    # Print all forms
    print("\n=== All forms ===")
    for form in soup.find_all('form'):
        print(f"  action={form.get('action')} | method={form.get('method')} | class={form.get('class')}")
    
    # Print all search related elements
    print("\n=== Search related elements ===")
    for el in soup.find_all(class_=re.compile(r'search', re.I)):
        print(f"  {el.name} | class={el.get('class')} | id={el.get('id')} | href={el.get('href','')}")
    
    # Also check for search in IDs
    for el in soup.find_all(id=re.compile(r'search', re.I)):
        print(f"  ID MATCH: {el.name} | id={el.get('id')} | class={el.get('class')}")
        
    # Print current URL
    print(f"\nCurrent URL: {driver.current_url}")
    
finally:
    driver.quit()

=== All inputs ===
  type=text | id=None | class=['search_input'] | name=title | placeholder=Search...
  type=checkbox | id=dropdown-list-News | class=None | name=None | placeholder=None
  type=checkbox | id=dropdown-list-EXCLUSIVE | class=None | name=None | placeholder=None
  type=checkbox | id=dropdown-list-FINTRACKR | class=None | name=None | placeholder=None
  type=checkbox | id=dropdown-list-REPORTS | class=None | name=None | placeholder=None
  type=checkbox | id=dropdown-list-SNIPPETS | class=None | name=None | placeholder=None
  type=checkbox | id=dropdown-list-OTHERS | class=None | name=None | placeholder=None
  type=text | id=None | class=['w-100', 'subscribe-text-head'] | name=name | placeholder=Your Name
  type=email | id=None | class=['w-100', 'subscribe-text-head'] | name=email | placeholder=Email address
  type=hidden | id=None | class=None | name=check_for_captcha | placeholder=None
  type=text | id=None | class=['w-100', 'subscribe-text-head'] | name=name | placeholder=

In [7]:
driver = init_driver()
try:
    driver.get("https://entrackr.com")
    time.sleep(3)
    # Print all input elements and buttons on homepage
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    print("=== All input elements ===")
    for inp in soup.find_all('input'):
        print(f"  type={inp.get('type')} | id={inp.get('id')} | class={inp.get('class')} | name={inp.get('name')}")
    print("\n=== All search-related elements ===")
    for el in soup.find_all(class_=re.compile(r'search', re.I)):
        print(f"  {el.name}.{el.get('class')} | id={el.get('id')}")
finally:
    driver.quit()

=== All input elements ===
  type=text | id=None | class=['search_input'] | name=title
  type=checkbox | id=dropdown-list-News | class=None | name=None
  type=checkbox | id=dropdown-list-EXCLUSIVE | class=None | name=None
  type=checkbox | id=dropdown-list-FINTRACKR | class=None | name=None
  type=checkbox | id=dropdown-list-REPORTS | class=None | name=None
  type=checkbox | id=dropdown-list-SNIPPETS | class=None | name=None
  type=checkbox | id=dropdown-list-OTHERS | class=None | name=None
  type=text | id=None | class=['w-100', 'subscribe-text-head'] | name=name
  type=email | id=None | class=['w-100', 'subscribe-text-head'] | name=email
  type=hidden | id=None | class=None | name=check_for_captcha
  type=text | id=None | class=['w-100', 'subscribe-text-head'] | name=name
  type=email | id=None | class=['w-100', 'subscribe-text-head'] | name=email
  type=hidden | id=None | class=None | name=check_for_captcha
  type=text | id=None | class=['input-field-name'] | name=name
  type=email 